# 3) Multi-period dispatch: ramp limits and battery storage

## Objectives

At the end of this notebook you should be able to:

- extend the two-plant LP of lesson 2 to **24 hours** in Linopy, with variables indexed by time
- show that without coupling constraints the 24-hour LP is just **24 merit orders**
- add **ramp limits** and explain why they couple the hours, raise cost, cause curtailment and create prices that are **no plant's marginal cost**
- decompose such a price with the **KKT conditions** (Maths-3)
- add a **battery** with a state-of-charge equation, and measure what it is worth
- **reproduce PyPSA's objective and every hourly price** with your own Linopy model — the week-9 checkpoint skill

Lesson 2 showed that the merit order solves an LP. This lesson shows where the merit order stops being enough: as soon as decisions in one hour constrain the next, only the full LP finds the cheapest plan. The day is **synthetic** but Croatia-sized.

In [1]:
import logging
import warnings

import linopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypsa
import xarray as xr

logging.getLogger("pypsa").setLevel(logging.ERROR)
logging.getLogger("linopy").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

### A sunny day

Demand rises from 1,900 MW at night to 2,300 MW at midday with an evening peak; 2,200 MW of solar and 400 MW of run-of-river hydro offer free energy; three thermal plants fill the gap. The coal plant is cheap but slow: it can change its output by at most **60 MW per hour**.

In [2]:
T = 24
hours = np.arange(T)
load = 1900 + 400 * np.sin(np.pi * (hours - 6) / 12).clip(0) + 250 * np.exp(-((hours - 19) / 2.0) ** 2)   # MW
solar = 2200 * 0.8 * np.clip(np.sin(np.pi * (hours - 6) / 13), 0, None)                                   # MW
free = solar + 400.0                                      # solar + run-of-river, offered at 0 EUR/MWh
plants = pd.DataFrame({"p_nom": [900, 900, 500], "mc": [60, 95, 150], "ramp": [60, 300, 500]},
                      index=pd.Index(["coal", "ccgt", "ocgt"], name="gen"))   # MW, EUR/MWh, MW/h
BATTERY = {"p_nom": 300, "e_nom": 1200, "eta": 0.95}       # MW, MWh, one-way efficiency
pd.DataFrame({"load_MW": load, "free_MW": free}).iloc[[0, 6, 9, 12, 16, 19, 23]].round(0)

    load_MW  free_MW
0    1900.0    400.0
6    1900.0    400.0
9    2183.0   1567.0
12   2300.0   2147.0
16   2126.0   1567.0
19   2150.0    400.0
23   1905.0    400.0

### One model, three versions

Build the LP once, with switches for the two coupling elements. Variables now carry a time index **t**: generation g[gen, t], free energy used f[t], and for the battery charge c[t], discharge d[t] and state of charge e[t] (MWh). The balance holds in every hour; its dual in hour t is that hour's price.

- **ramp limits**: −R_gen ≤ g[gen, t] − g[gen, t−1] ≤ R_gen for t ≥ 1
- **battery**: e[t] = e[t−1] + η·c[t] − d[t]/η, with e[−1] = e[23] (the battery ends the day as full as it started — *cyclic*)

`soc.roll(t=1)` gives e[t−1] with the wrap-around; `g.shift(t=1)` gives g[t−1] without it, and the mask skips hour 0. `output_flag=False` keeps HiGHS quiet.

In [3]:
def dispatch(ramps=True, battery=True):
    """Own Linopy model of the day. Returns the solved model."""
    m = linopy.Model()
    t = pd.RangeIndex(T, name="t")
    g = m.add_variables(lower=0, upper=xr.DataArray(np.tile(plants.p_nom.to_numpy(), (T, 1)), coords=[t, plants.index]),
                        name="g")
    f = m.add_variables(lower=0, upper=xr.DataArray(free, coords=[t]), name="free")
    supply = g.sum("gen") + f
    if battery:
        c = m.add_variables(lower=0, upper=BATTERY["p_nom"], coords=[t], name="charge")
        d = m.add_variables(lower=0, upper=BATTERY["p_nom"], coords=[t], name="discharge")
        e = m.add_variables(lower=0, upper=BATTERY["e_nom"], coords=[t], name="soc")
        m.add_constraints(e - e.roll(t=1) - BATTERY["eta"] * c + d / BATTERY["eta"] == 0, name="soc_balance")
        supply = supply + d - c
    m.add_constraints(supply == xr.DataArray(load, coords=[t]), name="balance")
    if ramps:
        r = xr.DataArray(plants.ramp.to_numpy(), coords=[plants.index])
        later = xr.DataArray(hours > 0, coords=[t])
        m.add_constraints(g - g.shift(t=1) <= r, name="ramp_up", mask=later)
        m.add_constraints(g.shift(t=1) - g <= r, name="ramp_down", mask=later)
    m.add_objective((g * xr.DataArray(plants.mc.to_numpy(), coords=[plants.index])).sum())
    m.solve(solver_name="highs", output_flag=False)
    return m


def summary(m):
    price = m.constraints["balance"].dual.to_numpy() + 0.0           # + 0.0 turns -0.0 into 0.0
    return {"cost_EUR": m.objective.value, "curtailed_MWh": float((free - m.variables["free"].solution).sum()),
            "min_price": price.min(), "max_price": price.max()}


models = {"no coupling": dispatch(ramps=False, battery=False),
          "ramp limits": dispatch(ramps=True, battery=False),
          "ramp limits + battery": dispatch(ramps=True, battery=True)}
pd.DataFrame({k: summary(m) for k, m in models.items()}).T.round(1)

                        cost_EUR  curtailed_MWh  min_price  max_price
no coupling            1820566.8            0.0       60.0       95.0
ramp limits            1955862.3         1322.7        0.0      150.0
ramp limits + battery  1837742.7           23.6        0.0       95.0

### Step 1: without coupling, 24 merit orders

With no ramp limits and no storage, nothing links one hour to the next, so the 24-hour LP falls apart into 24 independent one-hour problems — each solved by the merit order (lesson 2). Check it:

In [4]:
def merit_order_price(demand, free_available):
    """Price of one hour: the marginal cost of the unit that covers the last MW."""
    remaining = demand - free_available
    if remaining <= 0:
        return 0.0
    for gen, row in plants.sort_values("mc").iterrows():
        if remaining <= row.p_nom:
            return float(row.mc)
        remaining -= row.p_nom
    raise ValueError("demand exceeds capacity")


mo_prices = np.array([merit_order_price(load[t], free[t]) for t in hours])
lp_prices = models["no coupling"].constraints["balance"].dual.to_numpy()
assert np.allclose(mo_prices, lp_prices)
print("hourly LP prices = hourly merit-order prices in all 24 hours")
print(np.round(lp_prices + 0.0, 0))

hourly LP prices = hourly merit-order prices in all 24 hours
[95. 95. 95. 95. 95. 95. 95. 95. 60. 60. 60. 60. 60. 60. 60. 60. 60. 60.
 95. 95. 95. 95. 95. 95.]


Night hours cost 95 €/MWh (the CCGT is marginal), sunny hours 60 €/MWh (coal is marginal, swinging between 900 MW and 139 MW). That swing is the problem: real coal plants cannot do it.

### Step 2: ramp limits couple the hours

In [5]:
free_run = models["no coupling"].variables["g"].solution.sel(gen="coal").to_numpy()
ramped = models["ramp limits"]
coal = ramped.variables["g"].solution.sel(gen="coal").to_numpy()
price = ramped.constraints["balance"].dual.to_numpy() + 0.0
pd.DataFrame({"load": load, "free": free, "coal_unlimited": free_run, "coal_ramp_limited": coal,
              "curtailed": free - ramped.variables["free"].solution.to_numpy(), "price": price}, index=hours).round(0).T

                       0       1       2       3       4       5       6   \
load               1900.0  1900.0  1900.0  1900.0  1900.0  1900.0  1900.0   
free                400.0   400.0   400.0   400.0   400.0   400.0   400.0   
coal_unlimited      900.0   900.0   900.0   900.0   900.0   900.0   900.0   
coal_ramp_limited   900.0   900.0   900.0   900.0   900.0   840.0   780.0   
curtailed             0.0     0.0     0.0     0.0     0.0     0.0     0.0   
price                95.0    95.0    95.0    95.0    95.0    95.0    95.0   

                       7       8       9   ...      14      15      16  \
load               2004.0  2100.0  2183.0  ...  2247.0  2187.0  2126.0   
free                821.0  1218.0  1567.0  ...  2046.0  1848.0  1567.0   
coal_unlimited      900.0   882.0   616.0  ...   201.0   339.0   559.0   
coal_ramp_limited   720.0   660.0   600.0  ...   439.0   499.0   559.0   
curtailed             0.0     0.0     0.0  ...   238.0   160.0     0.0   
price           

The 60 MW/h limit changes the whole day:

- **Coal starts ramping down at 05:00** — before the sun rises — because it needs hours to get low enough for midday. The LP sees the whole day at once and plans ahead; a merit order, hour by hour, cannot.
- It still cannot get below 379 MW, so in the sunny hours there is more free energy than demand: **1,323 MWh of solar is curtailed** and the price is 0 €/MWh.
- In the evening coal cannot ramp up fast enough, so the 150 €/MWh OCGT runs at 19:00.
- The day costs 135,295 € more (+7.4 %).

And look at 16:00: the price is **65 €/MWh — not the marginal cost of any plant**. Why?

### Decomposing the price with KKT

Coal runs between its limits at 16:00, so the KKT stationarity condition of Maths-3 holds for it. With ramp multipliers μ⁺ₜ, μ⁻ₜ ≥ 0 for the up and down limits at hour t (Linopy reports the duals of "≤" constraints as negative numbers, so μ = −dual):

**λₜ = mc_coal + (μ⁺ₜ − μ⁺ₜ₊₁) − (μ⁻ₜ − μ⁻ₜ₊₁)**

One more MW of coal at hour t tightens the ramp from t−1 and loosens the ramp to t+1 — and both are worth money:

In [6]:
mu_up = -ramped.constraints["ramp_up"].dual.sel(gen="coal").fillna(0).to_numpy()
mu_dn = -ramped.constraints["ramp_down"].dual.sel(gen="coal").fillna(0).to_numpy()
mu_up_next, mu_dn_next = np.r_[mu_up[1:], 0], np.r_[mu_dn[1:], 0]
reconstructed = plants.loc["coal", "mc"] + (mu_up - mu_up_next) - (mu_dn - mu_dn_next)
between_limits = (coal > 1e-6) & (coal < plants.loc["coal", "p_nom"] - 1e-6)
assert np.allclose(reconstructed[between_limits], price[between_limits])
print(f"KKT reconstruction matches the price in all {between_limits.sum()} hours where coal is between its limits")
print(f"16:00: price {price[16]:.0f} = mc 60 + (mu_up[16] {mu_up[16]:.0f} - mu_up[17] {mu_up[17]:.0f})")

KKT reconstruction matches the price in all 17 hours where coal is between its limits
16:00: price 65 = mc 60 + (mu_up[16] 180 - mu_up[17] 175)


At 16:00 coal is ramping up as fast as it can, and the ramp limits of the following hours are binding too. One more MW of coal at 16:00 lets coal be one MW higher at 17:00 as well, which saves expensive generation later; that is worth 5 €/MWh on top of coal's 60. In multi-period models, **prices carry information about the future** — a property no single-hour merit order has.

### Step 3: a battery

In [7]:
full = models["ramp limits + battery"]
d = full.variables["discharge"].solution.to_numpy()
c = full.variables["charge"].solution.to_numpy()
soc = full.variables["soc"].solution.to_numpy()
p_full = full.constraints["balance"].dual.to_numpy() + 0.0
eta = BATTERY["eta"]
assert np.isclose(soc[0], soc[-1] + eta * c[0] - d[0] / eta), "cyclic: hour 0 starts from the state of hour 23"
assert (soc <= BATTERY["e_nom"] + 1e-6).all()
saving = models["ramp limits"].objective.value - full.objective.value
profit = float((p_full * (d - c)).sum())
print(f"battery: charged {c.sum():,.0f} MWh, discharged {d.sum():,.0f} MWh "
      f"(round-trip efficiency {d.sum() / c.sum():.3f} = 0.95 x 0.95)")
print(f"system cost saving {saving:,.0f} EUR/day; battery market profit at the LP prices {profit:,.0f} EUR/day")
pd.DataFrame({"charge": c, "discharge": d, "soc_MWh": soc, "price": p_full}, index=hours).round(0).T + 0.0

battery: charged 1,263 MWh, discharged 1,140 MWh (round-trip efficiency 0.902 = 0.95 x 0.95)
system cost saving 118,120 EUR/day; battery market profit at the LP prices 108,300 EUR/day


              0     1     2     3     4     5     6     7     8     9   ...  \
charge       0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
discharge  300.0   0.0   0.0  67.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
soc_MWh     70.0  70.0  70.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
price       95.0  95.0  95.0  95.0  95.0  95.0  95.0  95.0  95.0  95.0  ...   

               14      15      16      17      18     19     20     21     22  \
charge      238.0   160.0     0.0     0.0     0.0    0.0    0.0    0.0    0.0   
discharge     0.0     0.0     0.0     0.0    36.0  300.0    0.0  137.0  300.0   
soc_MWh    1048.0  1200.0  1200.0  1200.0  1162.0  846.0  846.0  702.0  386.0   
price         0.0     0.0    65.0    95.0    95.0   95.0   95.0   95.0   95.0   

              23  
charge       0.0  
discharge    0.0  
soc_MWh    386.0  
price       95.0  

[4 rows x 24 columns]

The battery charges from the solar surplus at midday (price 0) and discharges at full power at 19:00, replacing the OCGT: the evening price drops from 150 to 95 €/MWh and curtailment from 1,323 MWh to 24 MWh. It saves the system 118,120 € in this one day.

Notice that the rest of its discharge is spread over seemingly random hours — midnight, 03:00, late evening. Once the peak is covered, every remaining hour costs 95 €/MWh, so it does not matter *when* the battery delivers the rest of its energy: the LP has many equally cheap schedules and the solver returns one of them (degeneracy, Maths-2). The cost and the prices are unique; this particular schedule is not. Real storage operators break such ties with secondary goals, such as keeping energy in reserve.

Its own market profit — buying at the hourly price and selling at the hourly price — is smaller than the saving it creates for the system. Part of the value goes to consumers through lower evening prices. This gap between social value and private revenue is why storage investment is a policy question (Module 05).

In [8]:
fig, axes = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
for name, m in models.items():
    axes[0].step(hours, m.constraints["balance"].dual.to_numpy() + 0.0, where="mid", label=name, lw=2 if "battery" in name else 1.2)
axes[0].set(ylabel="price (EUR/MWh)", title="Hourly prices: coupling constraints change the price pattern")
axes[0].legend(fontsize=8)
axes[1].plot(hours, free_run, label="coal, no ramp limit")
axes[1].plot(hours, coal, label="coal, 60 MW/h ramp limit")
axes[1].fill_between(hours, 0, soc, alpha=0.3, label="battery state of charge (MWh)")
axes[1].set(xlabel="hour of day", ylabel="MW  /  MWh")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

### Validation: the same three models in PyPSA

PyPSA writes exactly these constraints for you: `ramp_limit_up`/`ramp_limit_down` (as a fraction of `p_nom`, applied between consecutive snapshots) and a `StorageUnit` with `efficiency_store`, `efficiency_dispatch`, `max_hours` and `cyclic_state_of_charge`. If your model is right, the objective and **every hourly price** must match:

In [9]:
def pypsa_day(ramps=True, battery=True):
    n = pypsa.Network()
    n.set_snapshots(hours)
    n.add("Bus", "HR")
    n.add("Load", "load", bus="HR", p_set=load)
    n.add("Generator", "free", bus="HR", p_nom=free.max(), p_max_pu=free / free.max(), marginal_cost=0)
    for gen, row in plants.iterrows():
        limits = {"ramp_limit_up": row.ramp / row.p_nom, "ramp_limit_down": row.ramp / row.p_nom} if ramps else {}
        n.add("Generator", gen, bus="HR", p_nom=row.p_nom, marginal_cost=row.mc, **limits)
    if battery:
        n.add("StorageUnit", "battery", bus="HR", p_nom=BATTERY["p_nom"], max_hours=BATTERY["e_nom"] / BATTERY["p_nom"],
              efficiency_store=BATTERY["eta"], efficiency_dispatch=BATTERY["eta"], cyclic_state_of_charge=True)
    n.optimize(solver_name="highs", log_to_console=False, progress=False)
    return n


rows = {}
for (name, m), flags in zip(models.items(), [(False, False), (True, False), (True, True)]):
    n = pypsa_day(*flags)
    gap = np.abs(m.constraints["balance"].dual.to_numpy() - n.buses_t.marginal_price["HR"].to_numpy()).max()
    rows[name] = {"own_cost": m.objective.value, "pypsa_cost": n.objective, "max_price_difference": gap}
    assert np.isclose(m.objective.value, n.objective, rtol=1e-9) and gap < 1e-6
pd.DataFrame(rows).T

                           own_cost    pypsa_cost  max_price_difference
no coupling            1.820567e+06  1.820567e+06                   0.0
ramp limits            1.955862e+06  1.955862e+06                   0.0
ramp limits + battery  1.837743e+06  1.837743e+06                   0.0

Identical objectives and prices that differ by less than 10⁻⁶ €/MWh in every hour. You now know exactly what PyPSA's `ramp_limit_up` and `StorageUnit` mean mathematically — including conventions that are easy to get wrong: ramp limits are fractions of `p_nom` and only apply *between* snapshots, and a cyclic battery ends where it started. Reproducing a tool's numbers with your own formulation is the strongest test that you understand it (and it is the week-9 checkpoint).

## Check your understanding

**Coupling**

1. Why does coal start ramping down at 05:00, before the sun rises? What would a myopic operator, deciding one hour at a time, do instead — and what would it cost?
2. Name two other constraints, besides ramps and storage, that link hours together in a dispatch model.

**Prices and storage**

1. Using the KKT decomposition, explain in words why the 16:00 price is above coal's marginal cost.
2. The battery's profit at market prices is smaller than the system saving it creates. Who receives the difference? What would happen to the battery's profit if ten batteries were built?